In [1]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

In [12]:
time = datetime.now(ZoneInfo("Asia/Kolkata"))

expiry = (time + timedelta(days=60)).replace(
    hour=23,
    minute=59,
    second=59,
    microsecond=999999
)

print(time)

print(expiry)

print((expiry-time)>timedelta(days=0))
print(timedelta(days=0))

2026-06-09 15:07:47.715442+05:30
2026-08-08 23:59:59.999999+05:30
True
0:00:00


In [57]:
def listify(x: str) -> list:
    if type(x)==type([]):
        raise TypeError("Input Is a List!!")
    a = x[1:-1]
    a = a.split(sep=", ")
    ans = []
    temp = []
    flag = False
    for i in range(len(a)):
        if flag:
            if "]" in a[i]:
                temp.append(a[i][:-1])
                temp = "["+", ".join(temp)+"]"
                ans.append(listify(temp))
                temp = []
                flag = False
                continue
            temp.append(a[i])
            continue
        if a[i].isnumeric():
            a[i] = int(a[i])
            ans.append(a[i])
            continue
        if "[" in a[i]:
            temp.append(a[i][1:])
            flag = True
            continue

    return ans

In [60]:
a = [1,2,3,4,[1,2,[2]]]

print(a)

b = listify(str(a))
print(b)

# print(str(a))
# a = str(a)

# print(a)

# a = a[1:-1]
# a = a.split(sep=", ")
# print(a)
# print(a[0].isnumeric())



[1, 2, 3, 4, [1, 2, [2]]]
[1, 2, 3, 4, [1, 2]]


In [ ]:
import json

a = [1,3,2,[1,2,3,[]]]

print(json.dumps(a))

json.loads

[1, 3, 2, [1, 2, 3, []]]


{
    cash_balance: 125000,

    rtgs_balance: 340000,

    cash_days:[
        {
            date:"2026-06-12",

            closing_balance:15000,

            transactions:[
                {
                    entity_name:"ABC Jewellers",

                    type:"SALE",

                    cash:10000,

                    created_at:
                        "2026-06-12T11:30:00"
                },

                {
                    entity_name:"XYZ Gold",

                    type:"PURCHASE",

                    cash:5000,

                    created_at:
                        "2026-06-12T16:20:00"
                }
            ]
        }
    ],

    rtgs_days:[
        {
            date:"2026-06-12",

            closing_balance:25000,

            transactions:[
                {
                    entity_name:"PQR Jewellery",

                    type:"SALE",

                    cash:25000,

                    created_at:
                        "2026-06-12T17:00:00"
                }
            ]
        }
    ]
}

@socketio.on("getCashReport")
def getCashReport(data):
    token = data.get("token")

    if not token:
        emit("error", {"msg": "no token"})
        return
    db = SessionLocal()
    try:
        decoded = decode_token(token)
        current_user = decoded["sub"]
        user = db.query(models.Users).filter_by(username=current_user).first()
        current_user_id = user.id
    except Exception as e:
        print("Error",e)
        emit("error", {"msg": "invalid token"})
        return
    try:
        cash_ob = db.query(models.Settings).filter_by(key="Cash Opening Balance").first()
        rtgs_ob = db.query(models.Settings).filter_by(key="Rtgs Opening Balance").first()
        
        cash_id = db.query(models.Item).filter_by(name="Cash Gold").first().id
        rtgs_id = db.query(models.Item).filter_by(name="Rtgs Gold").first().id
        
        cash_items = db.query(models.TransactionItem).filter(
            models.TransactionItem.item_id == cash_id,
            models.TransactionItem.delete_bool == False,
        ).all()

        cash_total = 0
        d = {}
        prev = None
        for item in cash_items:
            date = ""
            if prev!=item.transaction_id:
                date = db.query(models.Transaction).filter_by(id=item.transaction_id).first().created_at.date()
                date = str(date)
            prev = item.transaction_id
            if date not in d:
                d[date] = [item.id]
            else:
                d[date].append(item.id)
            if item.type=="SALE":
                cash_total += item.cash
            else:
                cash_total -= item.cash
        cash_list = []

        for key,value in d.items():
            print(key,value)
            date_dict = {}
            date_dict["date"] = key
            temp_list = []
            date_dict["closing_balance"] = 0
            for i in range(len(value)):
                item = db.query(models.TransactionItem).filter_by(id=value[i]).first()
                transaction = db.query(models.Transaction).filter_by(id = item.transaction_id).first()
                entity_name = db.query(models.Entities).filter_by(id=transaction.entity_id).first().name
                temp_list.append({
                    "entity_name": entity_name,
                    "type": item.type.name,

                    "cash": item.cash,

                    "created_at": str(transaction.created_at)
                })
                if item.type=="SALE":
                    date_dict["closing_balance"] += item.cash
                else:
                    date_dict["closing_balance"] -= item.cash
            date_dict["transactions"] = temp_list
            cash_list.append(date_dict)
        print(cash_list)
            
            
            
